In [3]:
# Import libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('stopwords')

# Load the dataset
df = pd.read_csv('/content/Laptop_Train_v2.csv')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


UnicodeDecodeError: ignored

In [ ]:
df

,id,Sentence,Aspect Term,polarity,from,to
0,2339,I charge it at night and skip taking the cord ...,cord,neutral,41,45
1,2339,I charge it at night and skip taking the cord ...,battery life,positive,74,86
2,1316,The tech guy then said the service center does...,service center,negative,27,41
3,1316,The tech guy then said the service center does...,"""sales"" team",negative,109,121
4,1316,The tech guy then said the service center does...,tech guy,neutral,4,12
...,...,...,...,...,...,...
2353,2272,We also use Paralles so we can run virtual mac...,Windows Server Enterprise 2003,neutral,104,134
2354,2272,We also use Paralles so we can run virtual mac...,Windows Server 2008 Enterprise,neutral,140,170
2355,848,"How Toshiba handles the repair seems to vary, ...",repair,conflict,24,30
2356,848,"How Toshiba handles the repair seems to vary, ...",repair,positive,130,136


In [ ]:
map_polarity = {'neutral': 2, 'positive': 1, "negative": 0} # map data

df["polarity"] = df["polarity"].map(map_polarity)

In [ ]:
df=df.drop('from',axis=1)

In [ ]:
df=df.drop('to',axis=1)

In [ ]:
df.shape

(2358, 4)

In [ ]:
df=df.dropna()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2313 entries, 0 to 2357
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           2313 non-null   int64  
 1   Sentence     2313 non-null   object 
 2   Aspect Term  2313 non-null   object 
 3   polarity     2313 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 90.4+ KB


In [ ]:
# Preprocess the data
def preprocess_text(Sentence):
    # Remove special characters and digits
    Sentence = re.sub('[^a-zA-Z]', ' ', Sentence)
    # Convert to lowercase
    Sentence = Sentence.lower()
    # Tokenize the text
    tokens = word_tokenize(Sentence)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    # Join the tokens back into a string
    Sentence = ' '.join(tokens)
    return Sentence

In [ ]:
# Apply the preprocessing function to the text column
Sentence = df['Sentence'].apply(preprocess_text)

In [ ]:

df.head()

,id,Sentence,Aspect Term,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,2.0
1,2339,I charge it at night and skip taking the cord ...,battery life,1.0
2,1316,The tech guy then said the service center does...,service center,0.0
3,1316,The tech guy then said the service center does...,"""sales"" team",0.0
4,1316,The tech guy then said the service center does...,tech guy,2.0


In [ ]:
y=df['polarity']

In [ ]:
Sentence

0         charge night skip taking cord good battery life
1         charge night skip taking cord good battery life
2       tech guy said service center exchange direct c...
3       tech guy said service center exchange direct c...
4       tech guy said service center exchange direct c...
                              ...                        
2352    also use paralles run virtual machines windows...
2353    also use paralles run virtual machines windows...
2354    also use paralles run virtual machines windows...
2356    toshiba handles repair seems vary folks indica...
2357    would like use different operating system alto...
Name: Sentence, Length: 2313, dtype: object

In [ ]:


# Define the BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Define a function to perform aspect-based sentiment analysis
def aspect_sentiment_analysis(text, aspect):
    # Tokenize the input text and aspect
    encoded_dict = tokenizer.encode_plus(
        text, aspect,
        add_special_tokens=True,
        max_length=64,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt'
    )

    # Make a prediction using the BERT model
    model.eval()
    with torch.no_grad():
        input_ids = encoded_dict['input_ids'].to(device)
        attention_mask = encoded_dict['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits.detach().cpu().numpy()

    # Determine the sentiment label (positive/negative) based on the prediction
    sentiment_label = np.argmax(logits)

    return sentiment_label

# Perform aspect-based sentiment analysis on each row of the CSV file
results = []
for index, row in df.iterrows():
    text = row['text']
    aspect = row['aspect']
    sentiment_label = aspect_sentiment_analysis(text, aspect)
    results.append(sentiment_label)

# Add the results to the CSV file
df['sentiment_label'] = results
df.to_csv('results.csv', index=False)

# Evaluate the performance of the model using F1 score and accuracy
true_labels = df['sentiment_label'].tolist()
predicted_labels = results
f1_score = f1_score(true_labels, predicted_labels)
accuracy_score = accuracy_score(true_labels, predicted_labels)
print('F1 score:', f1_score)
print('Accuracy:', accuracy_score)